In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
# ====================== IMPORTS ======================
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
import pickle

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [3]:
# ====================== PARAMETERS ======================
dataset_path = "/kaggle/input/19560-indian-takeaway-orders/restaurant-1-orders.csv"
top_n_items = 100   # top-selling items
lag_window = 30     # past 30 days used for prediction

# ====================== LOAD DATA ======================
df = pd.read_csv(dataset_path)
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True)

# ====================== AGGREGATE DAILY SALES ======================
daily_sales = df.groupby(['Order Date', 'Item Name'])['Quantity'].sum().unstack(fill_value=0)

# ====================== SELECT TOP ITEMS ======================
top_items = daily_sales.sum().sort_values(ascending=False).head(top_n_items).index.tolist()
daily_sales_top = daily_sales[top_items]

In [4]:
# ====================== FEATURE CREATION ======================
X, y = [], []
dates = daily_sales_top.index

for i in range(lag_window, len(daily_sales_top)):
    X.append(daily_sales_top.iloc[i - lag_window:i].values.flatten())  # flatten past lag_window days
    y.append(daily_sales_top.iloc[i].values)  # sales for next day

X = np.array(X)
y = np.array(y)

In [5]:
# ====================== TRAIN-TEST SPLIT ======================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

In [6]:
from tqdm import tqdm  # progress bar

# ====================== TRAIN MULTIOUTPUT XGBOOST WITH PROGRESS ======================
print("🚀 Training MultiOutput XGBoost model with progress...\n")

n_targets = y_train.shape[1]
models = []

for i in tqdm(range(n_targets), desc="Training progress", unit="model"):
    xgb = XGBRegressor(
        n_estimators=300,
        learning_rate=0.1,
        max_depth=5,
        subsample=0.9,
        colsample_bytree=0.8,
        random_state=42,
        verbosity=1
    )
    xgb.fit(X_train, y_train[:, i])
    models.append(xgb)

# Wrap trained models in MultiOutputRegressor-like structure
multi_model = MultiOutputRegressor(XGBRegressor())  # placeholder
multi_model.estimators_ = models

print("\n✅ Training complete!")


🚀 Training MultiOutput XGBoost model with progress...



Training progress: 100%|██████████| 100/100 [13:55<00:00,  8.35s/model]


✅ Training complete!


In [7]:
multi_model.fit(X_train, y_train)
print("\n✅ Training complete!")

# ====================== SAVE MODEL & ITEMS ======================
with open('multi_model.pkl', 'wb') as f:
    pickle.dump(multi_model, f)

with open('top_items.pkl', 'wb') as f:
    pickle.dump(top_items, f)

print("Model and top items saved successfully!")

Exception ignored on calling ctypes callback function: <bound method DataIter._next_wrapper of <xgboost.data.SingleBatchInternalIter object at 0x79bc1a4d6f10>>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/xgboost/core.py", line 589, in _next_wrapper
    def _next_wrapper(self, this: None) -> int:  # pylint: disable=unused-argument

KeyboardInterrupt: 


XGBoostError: [12:58:11] /workspace/src/data/iterative_dmatrix.cc:231: Check failed: accumulated_rows == Info().num_row_ (10440 vs. 20880) : 
Stack trace:
  [bt] (0) /usr/local/lib/python3.11/dist-packages/xgboost/lib/libxgboost.so(+0x3effba) [0x79bc1b893fba]
  [bt] (1) /usr/local/lib/python3.11/dist-packages/xgboost/lib/libxgboost.so(+0x3f7023) [0x79bc1b89b023]
  [bt] (2) /usr/local/lib/python3.11/dist-packages/xgboost/lib/libxgboost.so(+0x3f8858) [0x79bc1b89c858]
  [bt] (3) /usr/local/lib/python3.11/dist-packages/xgboost/lib/libxgboost.so(+0x3a2a07) [0x79bc1b846a07]
  [bt] (4) /usr/local/lib/python3.11/dist-packages/xgboost/lib/libxgboost.so(XGQuantileDMatrixCreateFromCallback+0x2b0) [0x79bc1b609c40]
  [bt] (5) /lib/x86_64-linux-gnu/libffi.so.8(+0x7e2e) [0x79bc89654e2e]
  [bt] (6) /lib/x86_64-linux-gnu/libffi.so.8(+0x4493) [0x79bc89651493]
  [bt] (7) /usr/lib/python3.11/lib-dynload/_ctypes.cpython-311-x86_64-linux-gnu.so(+0xa4d8) [0x79bc8a01c4d8]
  [bt] (8) /usr/lib/python3.11/lib-dy

In [8]:
# ====================== PREDICT NEXT DAY ======================
# Prepare input: last `lag_window` days
X_pred = daily_sales_top.tail(lag_window).values.flatten().reshape(1, -1)

# Predict
pred_next_day = multi_model.predict(X_pred)[0]
pred_next_day = np.round(pred_next_day).astype(int)  # round to integers

# Print results
print("\nPredicted Sales for next day:")
for item, qty in zip(top_items, pred_next_day):
    print(f"{item}: {qty}")


Predicted Sales for next day:
Plain Papadum: 1
Pilau Rice: 0
Plain Naan: 0
Garlic Naan: 0
Plain Rice: 0
Onion Bhajee: 0
Mango Chutney: 0
Chicken Tikka Masala: 0
Chapati: 0
Mint Sauce: 0
Bombay Aloo: 0
Peshwari Naan: 0
Mushroom Rice: 0
Keema Naan: 0
Meat Samosa: 0
Korma: 0
Onion Chutney: 0
Saag Aloo: 0
Korma - Chicken: 0
Butter Chicken: 0
Chicken Tikka (Main): 0
Madras: 0
Spicy Papadum: 0
Chicken Biryani: 0
Red Sauce: 0
Chicken Tikka: 0
Tandoori Mixed Grill: 0
Special Fried Rice: 0
Curry: 0
Aloo Gobi: 0
Lamb Biryani: 0
Tandoori Roti: 0
Sheek Kebab: 0
Mixed Starter: 0
Tarka Dall: 0
French Fries: 0
Saag Paneer: 0
Bhuna: 0
Madras - Chicken: 0
Chicken Tikka Biryani: 0
Chicken Tikka Jalfrezi: 0
Paratha: 0
Vegetable Rice: 0
Curry - Chicken: 0
Prawn Puree: 0
Tandoori Chicken (Main): 0
Keema Rice: 0
Chana Masala: 0
Dhansak: 0
Chicken Balti: 0
Lime Pickle: 0
Chicken Chaat: 0
Royal Paneer: 0
Chicken Shashlick: 0
Vegetable Biryani: 0
Chicken Tikka Chilli Masala: 0
Mushroom Bhajee: 0
Egg Rice: 0
C

In [9]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [11]:
# =========================
# 1. Load your dataset
# =========================
df = pd.read_csv('19560-indian-takeaway-orders/restaurant-1-orders.csv')  # replace with your CSV
df['Order Date'] = pd.to_datetime(df['Order Date'])

FileNotFoundError: [Errno 2] No such file or directory: '19560-indian-takeaway-orders/restaurant-1-orders.csv'

In [14]:
dataset_path = "/kaggle/input/19560-indian-takeaway-orders/restaurant-1-orders.csv"
df = pd.read_csv(dataset_path)
df['Order Date'] = pd.to_datetime(df['Order Date'])

ValueError: time data "31/07/2019 21:10" doesn't match format "%m/%d/%Y %H:%M", at position 46. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [15]:
import pandas as pd

dataset_path = "/kaggle/input/19560-indian-takeaway-orders/restaurant-1-orders.csv"
df = pd.read_csv(dataset_path)

# Convert 'Order Date' to datetime (day first)
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True)

print(df.head())


   Order Number          Order Date            Item Name  Quantity  \
0         16118 2019-08-03 20:25:00        Plain Papadum         2   
1         16118 2019-08-03 20:25:00     King Prawn Balti         1   
2         16118 2019-08-03 20:25:00          Garlic Naan         1   
3         16118 2019-08-03 20:25:00        Mushroom Rice         1   
4         16118 2019-08-03 20:25:00  Paneer Tikka Masala         1   

   Product Price  Total products  
0           0.80               6  
1          12.95               6  
2           2.95               6  
3           3.95               6  
4           8.95               6  


In [16]:
# Extract only the date part (ignore time) for daily aggregation
df['Order_Date_Only'] = df['Order Date'].dt.date

# Group by Item Name and date, then sum the quantity
daily_item_sales = df.groupby(['Item Name', 'Order_Date_Only'])['Quantity'].sum().reset_index()

# Optional: sort by item and date
daily_item_sales = daily_item_sales.sort_values(['Item Name', 'Order_Date_Only'])

print(daily_item_sales.head(10))


    Item Name Order_Date_Only  Quantity
0  Aloo Chaat      2016-03-07         2
1  Aloo Chaat      2016-03-09         1
2  Aloo Chaat      2016-03-10         1
3  Aloo Chaat      2016-03-27         1
4  Aloo Chaat      2016-04-18         1
5  Aloo Chaat      2016-04-22         1
6  Aloo Chaat      2016-05-13         1
7  Aloo Chaat      2016-05-19         1
8  Aloo Chaat      2016-06-10         1
9  Aloo Chaat      2016-07-17         2


In [19]:
# Extract only the date part (ignore time) for daily aggregation
df['Order_Date_Only'] = df['Order Date'].dt.date

# Group by Item Name and date, then sum the quantity
daily_item_sales = df.groupby(['Item Name', 'Order_Date_Only'])['Quantity'].sum().reset_index()

# Optional: sort by item and date
daily_item_sales = daily_item_sales.sort_values(['Item Name', 'Order_Date_Only'])

print(daily_item_sales.tail(10))


            Item Name Order_Date_Only  Quantity
44628  Vindaloo Sauce      2019-06-09         1
44629  Vindaloo Sauce      2019-06-11         1
44630  Vindaloo Sauce      2019-06-15         1
44631  Vindaloo Sauce      2019-06-16         1
44632  Vindaloo Sauce      2019-06-17         2
44633  Vindaloo Sauce      2019-06-28         1
44634  Vindaloo Sauce      2019-07-05         1
44635  Vindaloo Sauce      2019-07-16         1
44636  Vindaloo Sauce      2019-07-30         1
44637  Vindaloo Sauce      2019-08-02         1


In [18]:
# Pivot table: rows = dates, columns = items, values = quantity sold
item_daily_pivot = daily_item_sales.pivot(index='Order_Date_Only', columns='Item Name', values='Quantity')

# Fill missing values with 0 (no sales that day)
item_daily_pivot = item_daily_pivot.fillna(0)

print(item_daily_pivot.head())


Item Name        Aloo Chaat  Aloo Gobi  Aloo Methi  Baingan Hari Mirch  \
Order_Date_Only                                                          
2015-09-01              0.0        0.0         0.0                 0.0   
2015-09-08              0.0        0.0         0.0                 0.0   
2015-09-09              0.0        0.0         0.0                 0.0   
2015-09-29              0.0        0.0         0.0                 0.0   
2015-09-30              0.0        0.0         0.0                 0.0   

Item Name        Bengal Fish Biryani  Bengal Fish Karahi  Bengal Fry Fish  \
Order_Date_Only                                                             
2015-09-01                       0.0                 0.0              0.0   
2015-09-08                       0.0                 0.0              0.0   
2015-09-09                       0.0                 0.0              0.0   
2015-09-29                       0.0                 0.0              0.0   
2015-09-30         

In [20]:
# Convert Order Date to datetime (day first format)
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True)

# Create only date column (ignore time)
df['Order_Date_Only'] = df['Order Date'].dt.date

# Group by Item and Date, sum Quantity
daily_item_sales = df.groupby(['Item Name', 'Order_Date_Only'])['Quantity'].sum().reset_index()

# Pivot table: rows = dates, columns = items, values = quantity sold
item_daily_pivot = daily_item_sales.pivot(index='Order_Date_Only', columns='Item Name', values='Quantity').fillna(0)

# Optional: keep last 2 years of data
item_daily_pivot = item_daily_pivot.tail(730)

In [21]:
look_back = 90  # 1 year
X, y = [], []

for i in range(look_back, len(item_daily_pivot)):
    X.append(item_daily_pivot.iloc[i-look_back:i].values)  # past 365 days
    y.append(item_daily_pivot.iloc[i].values)              # today

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (640, 90, 248)
y shape: (640, 248)


In [22]:
# ----------------------
# 1. Flatten X to 2D
# ----------------------
X_flat = X.reshape(X.shape[0], X.shape[1]*X.shape[2])
print("X_flat shape:", X_flat.shape)  # (640, 90*248)

X_flat shape: (640, 22320)


In [23]:
# ----------------------
# 2. Train/test split
# ----------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_flat, y, test_size=0.2, shuffle=False
)


In [24]:
from tqdm import tqdm
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor

# ====================== TRAIN MULTI-OUTPUT XGBOOST WITH PROGRESS ======================
print("🚀 Training MultiOutput XGBoost model with progress...\n")

n_targets = y_train.shape[1]  # number of items to predict
models = []

for i in tqdm(range(n_targets), desc="Training progress", unit="model"):
    xgb = XGBRegressor(
        n_estimators=300,
        learning_rate=0.1,
        max_depth=5,
        subsample=0.9,
        colsample_bytree=0.8,
        random_state=42,
        verbosity=1
    )
    xgb.fit(X_train, y_train[:, i])
    models.append(xgb)

# Wrap trained models in MultiOutputRegressor-like structure
multi_model = MultiOutputRegressor(XGBRegressor())  # placeholder
multi_model.estimators_ = models

print("\n✅ Training complete!")


🚀 Training MultiOutput XGBoost model with progress...



Training progress: 100%|██████████| 248/248 [1:10:08<00:00, 16.97s/model]


✅ Training complete!


In [25]:
import pandas as pd
import numpy as np

# Target date
target_date = pd.to_datetime("2025-10-10")

# Last date in your dataset
last_date = pd.to_datetime(item_daily_pivot.index[-1])

# Number of days to predict ahead
days_ahead = (target_date - last_date).days

if days_ahead <= 0:
    raise ValueError("Target date is before the last date in your dataset!")

# Prepare initial input (last 'look_back' days)
X_input = item_daily_pivot.values[-look_back:].copy()  # shape: (look_back, n_items)

# Recursive prediction for each day
predictions = []

for _ in range(days_ahead):
    X_reshaped = X_input.reshape(1, look_back, -1)
    y_pred = multi_model.predict(X_reshaped)[0]  # predicted sales for next day
    predictions.append(y_pred)
    
    # Update input by rolling the window
    X_input = np.vstack([X_input[1:], y_pred])

# Convert the last prediction (for target date) to DataFrame
target_pred = pd.DataFrame(predictions[-1].reshape(1, -1), columns=item_daily_pivot.columns)
target_pred = target_pred.T
target_pred.columns = ['Predicted_Sales']
target_pred['Date'] = target_date

# Sort by predicted quantity
target_pred_sorted = target_pred.sort_values(by='Predicted_Sales', ascending=False)

print(target_pred_sorted.head(20))


ValueError: Feature shape mismatch, expected: 22320, got 90

In [30]:
# Target date
target_date = pd.to_datetime("2025-10-10")
last_date = pd.to_datetime(item_daily_pivot.index[-1])

days_ahead = (target_date - last_date).days
if days_ahead <= 0:
    raise ValueError("Target date is before the last date in your dataset!")

# Take last 'look_back' days
X_input = item_daily_pivot.values[-look_back:].copy()
n_items = X_input.shape[1]

# Flatten for XGBoost
X_input_flat = X_input.reshape(1, look_back * n_items)

predictions = []

for _ in range(days_ahead):
    y_pred = multi_model.predict(X_input_flat)[0]
    predictions.append(y_pred)
    
    # Update rolling window and flatten
    X_input = np.vstack([X_input[1:], y_pred])
    X_input_flat = X_input.reshape(1, look_back * n_items)

# Convert last prediction to DataFrame
target_pred = pd.DataFrame(predictions[-1].reshape(1, -1), columns=item_daily_pivot.columns)
target_pred = target_pred.T
target_pred.columns = ['Predicted_Sales']
target_pred['Date'] = target_date

# Sort by predicted sales
target_pred_sorted = target_pred.sort_values(by='Predicted_Sales', ascending=False)
print(target_pred_sorted.head(20))


KeyboardInterrupt: 

In [31]:
import pandas as pd
import numpy as np

# -----------------------------
# TARGET DATE
# -----------------------------
target_date = pd.to_datetime("2025-10-10")

# Number of items
n_items = item_daily_pivot.shape[1]

# Take last 'look_back' days as input
X_input = item_daily_pivot.values[-look_back:].copy()  # shape: (look_back, n_items)

# -----------------------------
# FLATTEN INPUT
# -----------------------------
X_input_flat = X_input.reshape(1, look_back * n_items)

# -----------------------------
# PREDICT TARGET DATE
# -----------------------------
y_pred = multi_model.predict(X_input_flat)[0]

# -----------------------------
# CONVERT TO DATAFRAME
# -----------------------------
target_pred = pd.DataFrame({
    "Item Name": item_daily_pivot.columns,
    "Predicted_Sales": y_pred
})

target_pred['Date'] = target_date

# Sort by predicted quantity
target_pred_sorted = target_pred.sort_values(by='Predicted_Sales', ascending=False)

# Show top 20 items
print(target_pred_sorted.head(20))


                Item Name  Predicted_Sales       Date
188         Plain Papadum        14.345240 2025-10-10
186            Pilau Rice         6.847050 2025-10-10
81            Garlic Naan         5.085577 2025-10-10
187            Plain Naan         4.825891 2025-10-10
98        Korma - Chicken         3.246195 2025-10-10
88             Keema Naan         3.222433 2025-10-10
165          Onion Bhajee         2.758676 2025-10-10
144         Mango Chutney         2.572051 2025-10-10
153            Mint Sauce         2.512931 2025-10-10
52   Chicken Tikka Masala         2.424879 2025-10-10
189            Plain Rice         2.310066 2025-10-10
138      Madras - Chicken         2.030023 2025-10-10
185         Peshwari Naan         1.829196 2025-10-10
16            Bombay Aloo         1.790599 2025-10-10
20         Butter Chicken         1.663525 2025-10-10
146           Meat Samosa         1.663392 2025-10-10
44   Chicken Tikka (Main)         1.636351 2025-10-10
163         Mushroom Rice   

In [32]:
import pickle

# Save model
with open("1multi_model.pkl", "wb") as f:
    pickle.dump(multi_model, f)

# Save item names
item_names = item_daily_pivot.columns.tolist()
with open("item_names.pkl", "wb") as f:
    pickle.dump(item_names, f)


In [29]:
import pickle

model_data = {
    "multi_model": multi_model,
    "look_back": look_back,
    "columns": item_daily_pivot.columns.tolist()
}

with open("multi_model.pkl", "wb") as f:
    pickle.dump(model_data, f)


In [27]:
with open("multi_model.pkl", "rb") as f:
    model_data = pickle.load(f)

multi_model = model_data["multi_model"]
look_back = model_data["look_back"]
columns = model_data["columns"]

# Prepare input in the same column order
X_input = item_daily_pivot[columns].values[-look_back:].reshape(1, look_back * len(columns))
y_pred = multi_model.predict(X_input)


In [ ]:
# ========================
# PREDICT ITEM SALES AND TOTAL PRICE FOR TARGET DATE
# ========================

import numpy as np
import pandas as pd
import pickle

# ------------------------
# LOAD TRAINED MODELS
# ------------------------
with open("multi_quantity.pkl", "rb") as f:
    multi_quantity = pickle.load(f)

with open("multi_price.pkl", "rb") as f:
    multi_price = pickle.load(f)

with open("item_names.pkl", "rb") as f:
    item_names = pickle.load(f)

# ------------------------
# PARAMETERS
# ------------------------
look_back = 7  # same as used during training
target_date = pd.to_datetime("2025-10-10").date()  # target date for prediction

# ------------------------
# LOAD PIVOT TABLES (or create from dataset if not saved)
# ------------------------
# quantity_pivot: rows=dates, columns=items, values=quantity
# price_pivot: rows=dates, columns=items, values=total price
# Assume already created during training
n_items = len(item_names)

# ------------------------
# PREPARE INPUT
# ------------------------
last_date = quantity_pivot.index[-1]
if isinstance(last_date, pd.Timestamp):
    last_date = last_date.date()

days_ahead = (target_date - last_date).days

if days_ahead <= 0:
    raise ValueError("Target date must be after the last date in the dataset!")

# Take last 'look_back' days as input
X_input_q = quantity_pivot.values[-look_back:].copy()  # for quantity
X_input_p = price_pivot.values[-look_back:].copy()     # for price

# ------------------------
# RECURSIVE PREDICTION
# ------------------------
for _ in range(days_ahead):
    # Flatten input to match training
    Xq_reshaped = X_input_q.reshape(1, look_back * n_items)
    Xp_reshaped = X_input_p.reshape(1, look_back * n_items)

    # Predict next day
    yq_pred = multi_quantity.predict(Xq_reshaped)[0]
    yp_pred = multi_price.predict(Xp_reshaped)[0]

    # Update rolling window
    X_input_q = np.vstack([X_input_q[1:], yq_pred])
    X_input_p = np.vstack([X_input_p[1:], yp_pred])

# ------------------------
# CREATE FINAL DATAFRAME
# ------------------------
target_pred = pd.DataFrame({
    "Item Name": item_names,
    "Predicted_Quantity": yq_pred,
    "Predicted_Total_Price": yp_pred
}).sort_values(by="Predicted_Quantity", ascending=False)

target_pred['Date'] = target_date

print(f"\nPredictions for {target_date}:")
print(target_pred.head(20))
